<a href="https://colab.research.google.com/github/raki-rankawat/stm32-thesis/blob/main/Model_Pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [277]:
# =====================================================
# Imports
# =====================================================

import os
import time
import tarfile
import random
import numpy as np
from pathlib import Path
from urllib.request import urlretrieve

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from google.colab import drive

In [278]:
# =====================================================
# Mount Google Drive
# =====================================================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [279]:
# =====================================================
# Reproducibility + Device Setup
# =====================================================

SEED = 41

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [280]:
# =====================================================
# Dataset Configuration
# =====================================================

VWW_URL = "https://www.silabs.com/public/files/github/machine_learning/benchmarks/datasets/vw_coco2014_96.tar.gz"

BASE_DIR = Path("/content/vww_work")
ARCHIVE_PATH = BASE_DIR / "vw_coco2014_96.tar.gz"
EXTRACT_DIR = BASE_DIR / "extracted"

MANIFEST_BASE_DIR = Path("/content/drive/My Drive/vww_fixed_split_manifests")

N_PER_CLASS = 5000
VAL_RATIO = 0.20

In [281]:
# =====================================================
# Download VWW Dataset
# =====================================================

def download_vww():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if ARCHIVE_PATH.exists() and ARCHIVE_PATH.stat().st_size > 0:
        print("✅ VWW archive already downloaded")
        return

    print("⬇️ Downloading VWW archive...")
    urlretrieve(VWW_URL, ARCHIVE_PATH)
    print("✅ Download complete:", ARCHIVE_PATH)

In [282]:
# =====================================================
# Safe Extract
# =====================================================

def safe_extract_tar(tar, path="."):
    path = Path(path).resolve()

    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(path)):
            raise RuntimeError("❌ Unsafe path detected in tar archive")

    tar.extractall(path)

In [283]:
# =====================================================
# Extract Dataset
# =====================================================

def extract_vww():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    if any(EXTRACT_DIR.iterdir()):
        print("✅ VWW already extracted")
        return

    print("📦 Extracting VWW archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        safe_extract_tar(tar, EXTRACT_DIR)

    print("✅ Extraction complete:", EXTRACT_DIR)

In [284]:
# =====================================================
# Locate Dataset Root
# =====================================================

def find_vww_root():
    for p in EXTRACT_DIR.rglob("person"):
        if p.is_dir() and (p.parent / "non_person").is_dir():
            return p.parent

    raise RuntimeError("❌ Could not find dataset directories")

In [285]:
# =====================================================
# Image Listing Helper
# =====================================================

def list_images(folder):
    exts = {".jpg", ".jpeg", ".png"}
    return sorted(
        [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts],
        key=lambda x: str(x)
    )

In [286]:
# =====================================================
# Manifest Helpers
# =====================================================

def manifest_paths():
    return {
        "train_person": MANIFEST_BASE_DIR / "train_person.txt",
        "val_person": MANIFEST_BASE_DIR / "val_person.txt",
        "train_non_person": MANIFEST_BASE_DIR / "train_non_person.txt",
        "val_non_person": MANIFEST_BASE_DIR / "val_non_person.txt",
    }

def manifests_ready():
    paths = manifest_paths()
    return all(p.exists() for p in paths.values())

def save_manifest(file_list, manifest_path):
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(manifest_path, "w") as f:
        for item in file_list:
            f.write(str(item) + "\n")

def load_manifest(manifest_path):
    with open(manifest_path, "r") as f:
        return [Path(line.strip()) for line in f if line.strip()]

In [287]:
# =====================================================
# Create Fixed Split Manifests Once
# =====================================================

def create_fixed_split_manifests(src_root):
    if manifests_ready():
        print("✅ Fixed split manifests already exist:", MANIFEST_BASE_DIR)
        return

    print("🧩 Creating fixed split manifests...")

    person_imgs = list_images(src_root / "person")
    nonperson_imgs = list_images(src_root / "non_person")

    if len(person_imgs) < N_PER_CLASS or len(nonperson_imgs) < N_PER_CLASS:
        raise ValueError(
            f"❌ Not enough images:\n"
            f"person: {len(person_imgs)} (need {N_PER_CLASS})\n"
            f"non_person: {len(nonperson_imgs)} (need {N_PER_CLASS})"
        )

    rng = random.Random(SEED)
    rng.shuffle(person_imgs)
    rng.shuffle(nonperson_imgs)

    person_sel = person_imgs[:N_PER_CLASS]
    nonperson_sel = nonperson_imgs[:N_PER_CLASS]

    def split_list(lst):
        n_val = int(len(lst) * VAL_RATIO)
        return lst[n_val:], lst[:n_val]   # train, val

    p_train, p_val = split_list(person_sel)
    n_train, n_val = split_list(nonperson_sel)

    paths = manifest_paths()
    save_manifest(p_train, paths["train_person"])
    save_manifest(p_val, paths["val_person"])
    save_manifest(n_train, paths["train_non_person"])
    save_manifest(n_val, paths["val_non_person"])

    print("✅ Fixed split manifests saved at:", MANIFEST_BASE_DIR)

In [288]:
# =====================================================
# Custom Dataset from Manifest
# =====================================================

class VWWManifestDataset(Dataset):
    def __init__(self, person_manifest, nonperson_manifest, transform=None):
        self.transform = transform
        self.samples = []

        person_files = load_manifest(person_manifest)
        nonperson_files = load_manifest(nonperson_manifest)

        for p in nonperson_files:
            self.samples.append((p, 0))

        for p in person_files:
            self.samples.append((p, 1))

        self.class_to_idx = {"non_person": 0, "person": 1}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [289]:
# =====================================================
# Prepare Dataset
# =====================================================

print("Step 1/4: Download")
download_vww()

print("Step 2/4: Extract")
extract_vww()

print("Step 3/4: Find root")
vww_root = find_vww_root()
print("✅ Dataset root:", vww_root)

print("Step 4/4: Create fixed manifests")
create_fixed_split_manifests(vww_root)

Step 1/4: Download
✅ VWW archive already downloaded
Step 2/4: Extract
✅ VWW already extracted
Step 3/4: Find root
✅ Dataset root: /content/vww_work/extracted/vw_coco2014_96
Step 4/4: Create fixed manifests
✅ Fixed split manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests


In [290]:
# =====================================================
# Data Loaders
# =====================================================

BATCH_SIZE = 64
IMG_SIZE = 96

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3
    ),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    )
])

paths = manifest_paths()

train_data = VWWManifestDataset(
    person_manifest=paths["train_person"],
    nonperson_manifest=paths["train_non_person"],
    transform=train_transform
)

val_data = VWWManifestDataset(
    person_manifest=paths["val_person"],
    nonperson_manifest=paths["val_non_person"],
    transform=eval_transform
)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Class mapping:", train_data.class_to_idx)
print("Train size:", len(train_data))
print("Val size:", len(val_data))

Class mapping: {'non_person': 0, 'person': 1}
Train size: 8000
Val size: 2000


In [291]:
# =====================================================
# MobileNetV2 Block
# =====================================================

class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()

        hidden_dim = in_channels * expand_ratio
        self.use_residual = stride == 1 and in_channels == out_channels

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(
                hidden_dim,
                hidden_dim,
                3,
                stride=stride,
                padding=1,
                groups=hidden_dim,
                bias=False
            ),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True),

            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        return self.block(x)

In [292]:
# =====================================================
# MobileNetV2 Model
# Must exactly match your trained checkpoint
# =====================================================

class VWW_MobileNetV2(nn.Module):
    def __init__(self):
        super().__init__()

        self.initial = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        self.features = nn.Sequential(
            InvertedResidual(32, 16, 1, 1),

            InvertedResidual(16, 24, 2, 6),
            InvertedResidual(24, 24, 1, 6),

            InvertedResidual(24, 32, 2, 6),
            InvertedResidual(32, 32, 1, 6),

            InvertedResidual(32, 64, 2, 6),
            InvertedResidual(64, 64, 1, 6),
        )

        self.final_conv = nn.Sequential(
            nn.Conv2d(64, 512, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU6(inplace=True)
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(512, 2)

        self._initialize_weights()

    def forward(self, x):
        x = self.initial(x)
        x = self.features(x)
        x = self.final_conv(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

In [293]:
# =====================================================
# Load Best Checkpoint
# IMPORTANT: use the exact checkpoint you want to prune
# =====================================================

checkpoint_path = "/content/drive/My Drive/Colab Notebooks/mobilenetv2_seed_85.pth"

model = VWW_MobileNetV2().to(device)
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)

print("✅ Loaded checkpoint:", checkpoint_path)
print("✅ Model device:", next(model.parameters()).device)

✅ Loaded checkpoint: /content/drive/My Drive/Colab Notebooks/mobilenetv2_seed_85.pth
✅ Model device: cuda:0


In [294]:
# =====================================================
# Evaluation Function
# =====================================================

def evaluate(model, loader, device):
    model.eval()

    correct = 0
    total = 0
    running_loss = 0.0

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            out = model(x)
            loss = criterion(out, y)

            running_loss += loss.item() * y.size(0)

            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return running_loss / total, 100.0 * correct / total

In [295]:
# =====================================================
# Accuracy Before Pruning
# =====================================================

base_loss, base_acc = evaluate(model, val_loader, device)
print(f"✅ Validation loss before pruning: {base_loss:.4f}")
print(f"✅ Validation accuracy before pruning: {base_acc:.2f}%")

✅ Validation loss before pruning: 0.4399
✅ Validation accuracy before pruning: 79.95%


In [296]:
# =====================================================
# Pruning Setup
# NOTE:
# Structured pruning is tricky for MobileNetV2 because of depthwise blocks,
# residual paths, and channel dependencies.
# Start with unstructured pruning first.
# =====================================================

PRUNE_AMOUNT = 0.30
PRUNE_TYPE = "unstructured"   # recommended: "unstructured"

layers_to_prune = []

for module in model.modules():
    if isinstance(module, nn.Conv2d):
        layers_to_prune.append((module, "weight"))

if PRUNE_TYPE == "structured":
    for layer, param in layers_to_prune:
        # This may break MobileNet efficiency badly and can hurt accuracy a lot.
        prune.ln_structured(layer, name=param, amount=PRUNE_AMOUNT, n=2, dim=0)
    print(f"✅ Structured pruning applied: {PRUNE_AMOUNT*100:.0f}%")
else:
    for layer, param in layers_to_prune:
        prune.l1_unstructured(layer, name=param, amount=PRUNE_AMOUNT)
    print(f"✅ Unstructured pruning applied: {PRUNE_AMOUNT*100:.0f}% on all conv layers")

✅ Unstructured pruning applied: 30% on all conv layers


In [297]:
# =====================================================
# Accuracy After Pruning (Before Fine-Tuning)
# =====================================================

pruned_loss, pruned_acc = evaluate(model, val_loader, device)
print(f"✅ Validation loss after pruning (before FT): {pruned_loss:.4f}")
print(f"✅ Validation accuracy after pruning (before FT): {pruned_acc:.2f}%")

✅ Validation loss after pruning (before FT): 0.6423
✅ Validation accuracy after pruning (before FT): 65.35%


In [298]:
# =====================================================
# Fine-Tune
# =====================================================

FT_EPOCHS = 5
FT_LR = 1e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=FT_LR)

start_time = time.time()

ft_train_losses = []
ft_train_accs = []

for epoch in range(1, FT_EPOCHS + 1):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        out = model(x)
        loss = criterion(out, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)

        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total

    ft_train_losses.append(epoch_loss)
    ft_train_accs.append(epoch_acc)

    print(
        f"FT Epoch {epoch}/{FT_EPOCHS} | "
        f"Train Loss: {epoch_loss:.4f} | "
        f"Train Acc: {epoch_acc:.2f}%"
    )

ft_minutes = (time.time() - start_time) / 60
print(f"Fine-tune time: {ft_minutes:.2f} minutes")

FT Epoch 1/5 | Train Loss: 0.5164 | Train Acc: 78.09%
FT Epoch 2/5 | Train Loss: 0.5141 | Train Acc: 77.90%
FT Epoch 3/5 | Train Loss: 0.5129 | Train Acc: 77.92%
FT Epoch 4/5 | Train Loss: 0.5104 | Train Acc: 78.67%
FT Epoch 5/5 | Train Loss: 0.5136 | Train Acc: 78.00%
Fine-tune time: 1.25 minutes


In [299]:
# =====================================================
# Accuracy After Fine-Tuning
# =====================================================

ft_loss, ft_acc = evaluate(model, val_loader, device)
print(f"✅ Validation loss after fine-tuning: {ft_loss:.4f}")
print(f"✅ Validation accuracy after fine-tuning: {ft_acc:.2f}%")

✅ Validation loss after fine-tuning: 0.4381
✅ Validation accuracy after fine-tuning: 79.55%


In [300]:
# =====================================================
# Make Pruning Permanent
# =====================================================

for layer, param in layers_to_prune:
    prune.remove(layer, param)

print("✅ Pruning reparameterization removed")


✅ Pruning reparameterization removed


In [301]:
# =====================================================
# Save Pruned + Fine-Tuned Model
# =====================================================

prune_tag = "st" if PRUNE_TYPE == "structured" else "unst"
prune_pct = int(PRUNE_AMOUNT * 100)

save_path = f"/content/drive/My Drive/Colab Notebooks/mobilenetv2_pruned_{prune_tag}_{prune_pct}pct.pth"
torch.save(model.state_dict(), save_path)

print("✅ Saved pruned + fine-tuned weights:", save_path)

✅ Saved pruned + fine-tuned weights: /content/drive/My Drive/Colab Notebooks/mobilenetv2_pruned_unst_30pct.pth


In [302]:
# =====================================================
# Final Summary
# =====================================================

print("\n" + "=" * 60)
print("MobileNetV2 Pruning Summary")
print("=" * 60)
print(f"Checkpoint used: {checkpoint_path}")
print(f"Prune type: {PRUNE_TYPE}")
print(f"Prune amount: {PRUNE_AMOUNT*100:.0f}%")
print(f"Base accuracy: {base_acc:.2f}%")
print(f"After pruning (before FT): {pruned_acc:.2f}%")
print(f"After fine-tuning: {ft_acc:.2f}%")
print(f"Fine-tune time: {ft_minutes:.2f} minutes")
print(f"Saved model: {save_path}")


MobileNetV2 Pruning Summary
Checkpoint used: /content/drive/My Drive/Colab Notebooks/mobilenetv2_seed_85.pth
Prune type: unstructured
Prune amount: 30%
Base accuracy: 79.95%
After pruning (before FT): 65.35%
After fine-tuning: 79.55%
Fine-tune time: 1.25 minutes
Saved model: /content/drive/My Drive/Colab Notebooks/mobilenetv2_pruned_unst_30pct.pth
